In [1]:
# 01_data_exploration

#Exploratory data analysis for the multicommodity dataset. Loads CSV(s), inspects shapes, missing values, distributions, and correlations.


In [2]:
# Setup: make sure project root is on sys.path
import os, sys
project_root = r"C:\Users\shiva\Documents\multicommodity-prices-indonesia"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams['figure.figsize'] = (10,6)
sns.set_style("whitegrid")


C:\Users\shiva\AppData\Roaming\Python\Python313\site-packages\seaborn\_statistics.py:32: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.3.1)
  from scipy.stats import gaussian_kde


In [3]:
from pathlib import Path

# Define output folders
out_dir = Path(project_root) / "outputs"
fig_dir = out_dir / "figures"
log_dir = out_dir / "logs"
chkpt_dir = out_dir / "checkpoints"

# Create folders if not exist
fig_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)
chkpt_dir.mkdir(parents=True, exist_ok=True)

print("Output folders ready:", out_dir)

data_csv_dir = Path(project_root) / "data" / "csv"
# pick the first csv file found
csv_files = sorted(data_csv_dir.glob("*.csv"))
if len(csv_files) == 0:
    raise FileNotFoundError(f"No CSV files found in {data_csv_dir}. Place your CSV there.")
csv_path = str(csv_files[0])
print("Using CSV:", csv_path)

df = pd.read_csv(csv_path, parse_dates=True)  # try parse_dates if date column exists
display(df.head())
print("Shape:", df.shape)


Output folders ready: C:\Users\shiva\Documents\multicommodity-prices-indonesia\outputs
Using CSV: C:\Users\shiva\Documents\multicommodity-prices-indonesia\data\csv\india_commodities_demo.csv


,date,rice,wheat,onion,tomato,potato,sugar,milk,egg,chicken,oil,chili
0,2017-01-08,42.26,36.56,18.40,16.92,20.36,30.16,51.53,5.99,124.42,125.44,85.19
1,2017-01-09,40.87,37.64,18.74,16.66,20.99,29.65,52.02,5.76,128.39,129.00,82.89
2,2017-01-10,40.80,39.89,19.90,16.87,22.36,30.68,53.77,5.66,133.02,132.15,83.17
3,2017-01-11,39.17,39.12,21.14,17.81,23.06,31.68,50.78,5.78,133.68,130.03,78.21
4,2017-01-12,37.31,36.62,21.13,17.96,23.62,31.90,48.17,5.68,126.53,125.85,74.98


Shape: (1460, 12)


In [4]:
desc = df.describe().T
desc.to_csv(log_dir / "data_summary.csv")

missing = df.isnull().sum().sort_values(ascending=False)
missing.to_csv(log_dir / "missing_values.csv")

print("Saved describe() and missing values into logs/")


Saved describe() and missing values into logs/


In [5]:
# try to detect a date column
date_cols = [c for c in df.columns if 'date' in c.lower() or 'time' in c.lower()]
if len(date_cols) > 0:
    date_col = date_cols[0]
    print("Detected date column:", date_col)
    df[date_col] = pd.to_datetime(df[date_col], errors='coerce')
    df = df.set_index(date_col).sort_index()
else:
    print("No date column auto-detected; using default RangeIndex.")
display(df.head())


Detected date column: date


,rice,wheat,onion,tomato,potato,sugar,milk,egg,chicken,oil,chili
date,,,,,,,,,,,
2017-01-08,42.26,36.56,18.40,16.92,20.36,30.16,51.53,5.99,124.42,125.44,85.19
2017-01-09,40.87,37.64,18.74,16.66,20.99,29.65,52.02,5.76,128.39,129.00,82.89
2017-01-10,40.80,39.89,19.90,16.87,22.36,30.68,53.77,5.66,133.02,132.15,83.17
2017-01-11,39.17,39.12,21.14,17.81,23.06,31.68,50.78,5.78,133.68,130.03,78.21
2017-01-12,37.31,36.62,21.13,17.96,23.62,31.90,48.17,5.68,126.53,125.85,74.98


In [6]:
df_num = df.select_dtypes(include=['number']).copy()
print("Numeric columns (commodities):", df_num.columns.tolist())
print("Numeric shape:", df_num.shape)
display(df_num.head())


Numeric columns (commodities): ['rice', 'wheat', 'onion', 'tomato', 'potato', 'sugar', 'milk', 'egg', 'chicken', 'oil', 'chili']
Numeric shape: (1460, 11)


,rice,wheat,onion,tomato,potato,sugar,milk,egg,chicken,oil,chili
date,,,,,,,,,,,
2017-01-08,42.26,36.56,18.40,16.92,20.36,30.16,51.53,5.99,124.42,125.44,85.19
2017-01-09,40.87,37.64,18.74,16.66,20.99,29.65,52.02,5.76,128.39,129.00,82.89
2017-01-10,40.80,39.89,19.90,16.87,22.36,30.68,53.77,5.66,133.02,132.15,83.17
2017-01-11,39.17,39.12,21.14,17.81,23.06,31.68,50.78,5.78,133.68,130.03,78.21
2017-01-12,37.31,36.62,21.13,17.96,23.62,31.90,48.17,5.68,126.53,125.85,74.98


In [7]:
cols_to_plot = df_num.columns.tolist()[:6]
ax = df_num[cols_to_plot].plot()
ax.set_title("Sample commodity time series")
ax.set_xlabel("Index / Date")
plt.legend(bbox_to_anchor=(1.05,1), loc='upper left')
plt.tight_layout()

plt.savefig(fig_dir / "time_series.png", dpi=300)
plt.close()
print("Saved -> outputs/figures/time_series.png")


Saved -> outputs/figures/time_series.png


In [8]:
df_num[cols_to_plot].hist(bins=30, figsize=(12,6))
plt.tight_layout()
plt.savefig(fig_dir / "histograms.png", dpi=300)
plt.close()
print("Saved -> outputs/figures/histograms.png")


Saved -> outputs/figures/histograms.png


In [9]:
corr = df_num.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=False, cmap='coolwarm', center=0)
plt.title("Feature correlation matrix")
plt.savefig(fig_dir / "correlation_heatmap.png", dpi=300)
plt.close()

corr.to_csv(log_dir / "correlation_matrix.csv")

print("Saved -> outputs/figures/correlation_heatmap.png")
print("Saved -> outputs/logs/correlation_matrix.csv")


Saved -> outputs/figures/correlation_heatmap.png
Saved -> outputs/logs/correlation_matrix.csv
